# IBM Ponder This — August 2026
## The blind wheel of buttons

> A wheel carries $N$ buttons, equidistant and perfectly symmetric. Each button is **on** or **off**,
> pressing one toggles it, and there is no indicator of any button's state. The starting position is
> arbitrary except that not all buttons are on. Each round the wheel is spun by an unseen (and possibly
> **malicious**) amount; the contestant then presses a subset of the positions $1,2,\dots,N$ (numbered
> from the top-left button, clockwise). The contestant wins the moment all $N$ buttons are on.
>
> Among strategies that guarantee a win in the **fewest rounds**, minimise the **sum of the pressed
> numbers**. That minimum is the *optimal solution sum*.

IBM's worked example for $N=2$ is `1,2,0,1,0,1,2,0`, of sum $7$.

---

## Answers

| | minimum rounds | **optimal solution sum** |
|---|---|---|
| $N = 2$ (given) | $3$ | $7$ |
| $N = 4$ | $15$ | $103$ |
| **$N = 8$** | $255$ | **$6279$** |
| **$N = 64$** (bonus $\star$) | $2^{64}-1$ | **$27{,}636{,}190{,}239{,}652{,}591{,}799{,}943$** |

The rest of this notebook derives those numbers, proves the structure, and checks everything against
exhaustive search on the cases small enough to brute force.

---

## 1. Setting up the algebra

Index the wheel positions by $\mathbb{Z}_N$ and let $s \in \mathbb{F}_2^N$ record the button states.
Since the target is *all on*, track the **deviation** $d = s + \mathbf{1}$; the contestant wins exactly when
$d = 0$, and the problem says $d \neq 0$ at the start.

Identify $\mathbb{F}_2^N$ with the group algebra

$$R \;=\; \mathbb{F}_2[x]/(x^N-1),\qquad d \;\longleftrightarrow\; \sum_i d_i x^i .$$

Pressing the subset $S$ after the wheel has been spun by $r$ changes the state by $x^r S$, so after $t$ rounds

$$d_t \;=\; d_0 \;+\; \sum_{i \le t} x^{r_i} S_i,$$

where the adversary chooses each $r_i$, adaptively, knowing everything.

**The crucial simplification.** $N$ is a power of two, and in characteristic $2$

$$x^N - 1 = (x-1)^N .$$

So with $u = 1+x$ we get $R \cong \mathbb{F}_2[u]/(u^N)$: a *local* ring whose only ideals form a single chain

$$R = I_0 \supset I_1 \supset \dots \supset I_N = 0, \qquad I_j = (u^j),\quad \dim I_j = N-j .$$

Every nonzero $S$ has a **valuation** $v(S) = \max\{\, j : u^j \mid S \,\}$, and each quotient $I_j/I_{j+1}$ is a single bit.

In [1]:
from typing import Dict, List, Tuple
import heapq

# A subset of buttons is a bitmask over indices i = 0..N-1;
# index i is the button PRINTED with the label i+1.

def labels(mask, N):
    return [i + 1 for i in range(N) if (mask >> i) & 1]

def cost(mask, N):
    # the quantity the puzzle asks us to minimise: sum of pressed labels
    return sum(i + 1 for i in range(N) if (mask >> i) & 1)

def rotate(mask, r, N):
    # the effect of the wheel having been spun by r
    r %= N
    return ((mask << r) | (mask >> (N - r))) & ((1 << N) - 1)

def divide_by_u(mask, N):
    # divide by u = 1+x, valid when the weight is even:
    # if s = (1+x)q then s_i = q_i + q_{i-1}, hence q_i = s_0 ^ ... ^ s_i
    q, acc = 0, 0
    for i in range(N):
        acc ^= (mask >> i) & 1
        q |= acc << i
    return q

def valuation(mask, N):
    # u-adic valuation; N for the zero element
    if mask == 0:
        return N
    v = 0
    while mask and bin(mask).count("1") % 2 == 0:
        mask = divide_by_u(mask, N)
        v += 1
    return v

def u_power_mask(j, N):
    # u^j = (1+x)^j; by Lucas' theorem C(j,i) is odd iff i is a submask of j
    return sum(1 << i for i in range(N) if (i & j) == i)

N = 8
for j in range(N):
    m = u_power_mask(j, N)
    print(f"u^{j} -> buttons {str(labels(m, N)):<28} valuation {valuation(m, N)}  cost {cost(m, N)}")

u^0 -> buttons [1]                          valuation 0  cost 1
u^1 -> buttons [1, 2]                       valuation 1  cost 3
u^2 -> buttons [1, 3]                       valuation 2  cost 4
u^3 -> buttons [1, 2, 3, 4]                 valuation 3  cost 10
u^4 -> buttons [1, 5]                       valuation 4  cost 6
u^5 -> buttons [1, 2, 5, 6]                 valuation 5  cost 14
u^6 -> buttons [1, 3, 5, 7]                 valuation 6  cost 16
u^7 -> buttons [1, 2, 3, 4, 5, 6, 7, 8]     valuation 7  cost 36


## 2. Why the adversary is much weaker than it looks

**Lemma.** For every $S$ and every rotation $r$,

$$x^r S \equiv S \pmod{I_{v(S)+1}} .$$

*Proof.* $x = 1+u$, so $x^r = (1+u)^r = 1 + (\text{multiple of } u)$. Hence
$x^r S = S + u\,(\cdots)\,S$, and $u S \in I_{v(S)+1}$. $\blacksquare$

Read that again — it is the whole puzzle. **The spin cannot touch the leading coefficient of a move.**
If the contestant plays a set of valuation $j$, then the level-$j$ bit of the state is flipped *with certainty*,
no matter how the wheel was spun; the spin only scrambles the levels **above** $j$.

So write the state in "level coordinates" $a_0, a_1, \dots, a_{N-1}$, where $a_j$ is the image of $d$ in $I_j/I_{j+1}$.
The game becomes:

> **State:** an unknown vector $a \in \mathbb{F}_2^N$, not all zero.
> **Move of valuation $j$:** flips $a_j$, and changes $a_{j+1},\dots,a_{N-1}$ adversarially.
> **Win:** $a = 0$.

All structure of the wheel has evaporated — only the valuation of each chosen subset matters.

In [2]:
# Empirically check the lemma on every subset of an 8-button wheel:
# rotating a move never changes it below level v(S)+1.
N = 8
bad = 0
for S in range(1, 1 << N):
    v = valuation(S, N)
    for r in range(N):
        diff = S ^ rotate(S, r, N)          # what the spin cost us
        if diff and valuation(diff, N) <= v:  # must live strictly higher
            bad += 1
print(f"subsets tested        : {(1 << N) - 1}")
print(f"lemma violations      : {bad}")

S = u_power_mask(3, N)  # valuation 3
print(f"\nexample S = {labels(S, N)} (valuation {valuation(S, N)})")
for r in range(N):
    d = S ^ rotate(S, r, N)
    lv = "-" if d == 0 else valuation(d, N)
    print(f"  spin {r}: S + x^{r}S = {str(labels(d, N)):<28} valuation {lv}")

subsets tested        : 255
lemma violations      : 0

example S = [1, 2, 3, 4] (valuation 3)
  spin 0: S + x^0S = []                           valuation -
  spin 1: S + x^1S = [1, 5]                       valuation 4
  spin 2: S + x^2S = [1, 2, 5, 6]                 valuation 5
  spin 3: S + x^3S = [1, 2, 3, 5, 6, 7]           valuation 4
  spin 4: S + x^4S = [1, 2, 3, 4, 5, 6, 7, 8]     valuation 7
  spin 5: S + x^5S = [2, 3, 4, 6, 7, 8]           valuation 4
  spin 6: S + x^6S = [3, 4, 7, 8]                 valuation 5
  spin 7: S + x^7S = [4, 8]                       valuation 4


## 3. The optimal length is $2^N-1$

Let $T(k)$ be the number of rounds needed when only levels $N-k, \dots, N-1$ can still be wrong.

**Upper bound.** Moves of valuation $\ge j+1$ never disturb level $j$. So: clear the higher levels
($T(k-1)$ rounds, assuming the coarse level is already $0$); if that failed, the coarse level must have been $1$,
so flip it with one valuation-$(N-k)$ move — which scrambles everything above it — and clear the higher levels again:

$$T(k) \;=\; 2\,T(k-1) + 1, \qquad T(0) = 0 \quad\Longrightarrow\quad T(N) = 2^N - 1 .$$

**Lower bound.** The same recursion runs backwards. The adversary keeps the level-$j$ bit alive: any winning
script must contain a valuation-$(N-k)$ move (otherwise the coarse bit is never flipped and the state with that
bit set survives forever), and the segments before and after it must each be complete solutions of the
$(k-1)$-level game, since the adversary re-randomises everything above after that flip. Hence
$T(k) \ge 2T(k-1)+1$, and $2^N-1$ is exactly optimal.

**The schedule is forced.** Equality in $T(k) = 2T(k-1)+1$ leaves no slack: unrolling the recursion, round $k$
of an optimal script must use a set whose valuation is exactly

$$v_k \;=\; (N-1) - \nu_2(k), \qquad k = 1,\dots,2^N-1,$$

the **ruler sequence**. Consequently valuation $j$ is used exactly $2^{\,j}$ times, and $\sum_j 2^j = 2^N-1$. ✓

Check against IBM's example: for $N=2$ the schedule is $v = (1,0,1)$, i.e. *both, one, both* — exactly `1,2,0,1,0,1,2,0`.

In [3]:
def valuation_schedule(N):
    # round k must use a subset of valuation exactly (N-1) - nu_2(k)
    sched = []
    for k in range(1, 1 << N):
        nu = (k & -k).bit_length() - 1   # 2-adic valuation of k
        sched.append(N - 1 - nu)
    return sched

for N in (2, 3, 4):
    print(f"N = {N}: schedule {valuation_schedule(N)}")

N = 8
sched = valuation_schedule(N)
print(f"\nN = 8: {len(sched)} rounds (2^8 - 1 = {2**8 - 1})")
print("first 20 valuations:", sched[:20])
for j in range(N):
    print(f"  valuation {j} used {sched.count(j):>3} times   (2^{j} = {2**j})")

N = 2: schedule [1, 0, 1]
N = 3: schedule [2, 1, 2, 0, 2, 1, 2]
N = 4: schedule [3, 2, 3, 1, 3, 2, 3, 0, 3, 2, 3, 1, 3, 2, 3]

N = 8: 255 rounds (2^8 - 1 = 255)
first 20 valuations: [7, 6, 7, 5, 7, 6, 7, 4, 7, 6, 7, 5, 7, 6, 7, 3, 7, 6, 7, 5]
  valuation 0 used   1 times   (2^0 = 1)
  valuation 1 used   2 times   (2^1 = 2)
  valuation 2 used   4 times   (2^2 = 4)
  valuation 3 used   8 times   (2^3 = 8)
  valuation 4 used  16 times   (2^4 = 16)
  valuation 5 used  32 times   (2^5 = 32)
  valuation 6 used  64 times   (2^6 = 64)
  valuation 7 used 128 times   (2^7 = 128)


## 4. Independent ground truth: exhaustive search for $N = 2$ and $N = 4$

Before trusting any of that theory, let us solve the game *by brute force*, with no algebra at all.

The contestant has no information, so a strategy is just a fixed script of subsets. Track the **knowledge state**:
the set of deviations the adversary can still keep alive. It starts as all $2^N-1$ nonzero states, and a move $S$ sends

$$K \;\longmapsto\; \{\, d + x^r S \;:\; d \in K,\; r \in \mathbb{Z}_N \,\} \setminus \{0\}$$

(the adversary picks the rotation *per surviving branch*, which is the strongest possible cheating). The contestant
wins when $K = \varnothing$. Dijkstra over these knowledge states with the lexicographic key
(rounds, sum-of-labels) gives the exact answer.

This is feasible for $N \le 4$ ($2^{15}$ knowledge states) and hopeless beyond — which is precisely why the algebra above matters.

In [4]:
def optimal_by_search(N):
    # exact (min rounds, then min label-sum) over the knowledge-state graph
    size = 1 << N
    start = ((1 << size) - 1) ^ 1          # every nonzero deviation possible

    moves = []
    for S in range(1, size):
        rots = sorted({rotate(S, r, N) for r in range(N)})
        moves.append((cost(S, N), S, rots))

    def step(K, rots):
        out, d, KK = 0, 0, K
        while KK:
            if KK & 1:
                for rr in rots:
                    e = d ^ rr
                    if e:
                        out |= 1 << e
            KK >>= 1
            d += 1
        return out

    INF = (10**9, 10**9)
    dist = {start: (0, 0)}
    prev = {}
    pq = [(0, 0, start)]
    while pq:
        rounds, total, K = heapq.heappop(pq)
        if (rounds, total) > dist.get(K, INF):
            continue
        if K == 0:
            path, cur = [], K
            while cur != start:
                cur, S = prev[cur]
                path.append(S)
            return rounds, total, path[::-1]
        for c, S, rots in moves:
            K2 = step(K, rots)
            cand = (rounds + 1, total + c)
            if cand < dist.get(K2, INF):
                dist[K2] = cand
                prev[K2] = (K, S)
                heapq.heappush(pq, (cand[0], cand[1], K2))
    raise RuntimeError("unsolvable")


def format_solution(rounds_masks, N):
    # IBM's answer format: pressed labels per round, rounds separated by 0
    out = []
    for m in rounds_masks:
        out.extend(str(l) for l in labels(m, N))
        out.append("0")
    return ",".join(out)


search_results = {}
for N in (2, 4):
    rounds, total, path = optimal_by_search(N)
    search_results[N] = (rounds, total)
    print(f"N = {N}")
    print(f"  minimum rounds      : {rounds}   (2^N - 1 = {2**N - 1})")
    print(f"  optimal solution sum: {total}")
    print(f"  valuations used     : {[valuation(m, N) for m in path]}")
    print(f"  a solution          : {format_solution(path, N)}")
    print()

assert search_results[2] == (3, 7)   # matches the example in the problem statement

N = 2
  minimum rounds      : 3   (2^N - 1 = 3)
  optimal solution sum: 7
  valuations used     : [1, 0, 1]
  a solution          : 1,2,0,1,0,1,2,0

N = 4
  minimum rounds      : 15   (2^N - 1 = 15)
  optimal solution sum: 103
  valuations used     : [3, 2, 3, 1, 3, 2, 3, 0, 3, 2, 3, 1, 3, 2, 3]
  a solution          : 1,2,3,4,0,1,3,0,1,2,3,4,0,1,2,0,1,2,3,4,0,1,3,0,1,2,3,4,0,1,0,1,2,3,4,0,1,3,0,1,2,3,4,0,1,2,0,1,2,3,4,0,1,3,0,1,2,3,4,0



The $N=2$ row reproduces IBM's example exactly (`1,2,0,1,0,1,2,0`, sum $7$), including their remark that using
button $2$ instead of button $1$ in the middle round would give $8$: the middle round only needs *some* single
button, so we take the cheapest label.

That is the pattern in general — the schedule fixes the **valuation** of each round, and within a round we are
free to take *any* subset of that valuation. So:

$$\boxed{\ \text{optimal sum} \;=\; \sum_{j=0}^{N-1} 2^{\,j}\, c(j),\qquad
c(j) \;=\; \min\{\, \mathrm{cost}(S) \;:\; v(S) = j \,\}\ }$$

where $\mathrm{cost}(S)$ is the sum of the printed labels of $S$. The rounds are independent, so each one
is minimised separately.

---

## 5. The cheapest subset of each valuation

Write $p = \operatorname{popcount}(j)$. The natural candidate is $u^j = (1+x)^j$ itself: by Lucas' theorem its
support is the set of **submasks of $j$**, so it has weight $2^p$, and the submasks of $j$ sum to $j\,2^{p-1}$:

$$\mathrm{cost}(u^j) \;=\; \sum_{i \subseteq j} (i+1) \;=\; 2^{p} + j\,2^{\,p-1} .$$

It turns out this is always optimal.

In [5]:
def c_formula(j):
    p = bin(j).count("1")
    return 1 if p == 0 else (1 << p) + j * (1 << (p - 1))


def min_cost_by_valuation_bruteforce(N):
    # exhaustive over all 2^N subsets
    best = {}
    for mask in range(1, 1 << N):
        v, c = valuation(mask, N), cost(mask, N)
        if v not in best or c < best[v][0]:
            best[v] = (c, mask)
    return best


for N in (2, 4, 8, 16):
    best = min_cost_by_valuation_bruteforce(N)
    for j in range(N):
        c, mask = best[j]
        assert c == c_formula(j),        (N, j, c, c_formula(j))
        assert mask == u_power_mask(j, N), (N, j, mask)
    print(f"N = {N:>2}  c(j) = {[best[j][0] for j in range(N)]}")

print("\nIn every case the brute-force optimum IS u^j, and c(j) = 2^p + j*2^(p-1).")
print("\nN = 8 in detail:")
best8 = min_cost_by_valuation_bruteforce(8)
for j in range(8):
    c, m = best8[j]
    print(f"  v = {j}: press {str(labels(m, 8)):<26} cost {c:>3}   used 2^{j} = {2**j:>3} times"
          f"   -> {2**j * c:>4}")

N =  2  c(j) = [1, 3]
N =  4  c(j) = [1, 3, 4, 10]
N =  8  c(j) = [1, 3, 4, 10, 6, 14, 16, 36]
N = 16  c(j) = [1, 3, 4, 10, 6, 14, 16, 36, 10, 22, 24, 52, 28, 60, 64, 136]

In every case the brute-force optimum IS u^j, and c(j) = 2^p + j*2^(p-1).

N = 8 in detail:
  v = 0: press [1]                        cost   1   used 2^0 =   1 times   ->    1
  v = 1: press [1, 2]                     cost   3   used 2^1 =   2 times   ->    6
  v = 2: press [1, 3]                     cost   4   used 2^2 =   4 times   ->   16
  v = 3: press [1, 2, 3, 4]               cost  10   used 2^3 =   8 times   ->   80
  v = 4: press [1, 5]                     cost   6   used 2^4 =  16 times   ->   96
  v = 5: press [1, 2, 5, 6]               cost  14   used 2^5 =  32 times   ->  448
  v = 6: press [1, 3, 5, 7]               cost  16   used 2^6 =  64 times   -> 1024
  v = 7: press [1, 2, 3, 4, 5, 6, 7, 8]   cost  36   used 2^7 = 128 times   -> 4608


### Why $u^j$ is optimal (proof, so the $N=64$ bonus is safe)

Brute force stops at $N=16$; the bonus needs $N=64$. Here is the argument, by halving the wheel.

Split the indices into a low half $L=\{0,\dots,\tfrac N2-1\}$ and a high half, writing $S = S_L + x^{N/2}S_H$ with
$\deg S_L, \deg S_H < N/2$. In characteristic two $(1+x)^{N/2} = 1 + x^{N/2}$, so

$$x^{N/2} = 1 + u^{N/2} \qquad\Longrightarrow\qquad S = (S_L + S_H) \;+\; u^{N/2}\,S_H .$$

It is convenient to prove the statement for a whole family of cost functions at once. For $\lambda \ge 0$ let

$$h_n(j;\lambda) \;=\; \min \Big\{ \textstyle\sum_{i \in \mathrm{supp}(Q)} (i + 1 + \lambda) \;:\; \deg Q < n,\ v(Q) = j \Big\},$$

so the quantity we want is $c(j) = h_N(j;0)$.

**Case $j < N/2$.** Then $u^{N/2}S_H \in I_{N/2} \subseteq I_{j+1}$, so $v(S)=j$ forces $v(S_L+S_H) = j$. Since the
cost is monotone under taking subsets and $\mathrm{supp}(S_L+S_H) \subseteq \mathrm{supp}(S_L)\cup\mathrm{supp}(S_H)$,
replacing $S$ by $S_L + S_H$ never costs more — the high half is pure waste. Hence
$h_N(j;\lambda) = h_{N/2}(j;\lambda)$.

**Case $j \ge N/2$.** A nonzero polynomial of degree $< N/2$ has valuation $< N/2$, so $v(S) \ge N/2$ forces
$S_L + S_H = 0$, i.e. $S_L = S_H =: Q$ and $S = u^{N/2}Q$ with $v(Q) = j - N/2$. Each index $i$ of $Q$ now appears
twice, at $i$ and at $i + N/2$:

$$\sum_{i\in\mathrm{supp}(Q)}\big[(i{+}1{+}\lambda) + (i{+}\tfrac N2{+}1{+}\lambda)\big]
\;=\; 2\sum_{i\in\mathrm{supp}(Q)}\Big(i+1+\lambda+\tfrac N4\Big),$$

so $h_N(j;\lambda) = 2\,h_{N/2}\big(j-\tfrac N2;\ \lambda + \tfrac N4\big)$.

Both branches strictly shrink $n$, the base case $n=1$ is $h_1(0;\lambda)=1+\lambda$, and unrolling the recursion
reproduces $2^p + j\,2^{p-1}$. The minimiser is unique and equals $u^j$. $\blacksquare$

The cell below checks the recursion against brute force, and then evaluates it symbolically at $N=64$ where
enumeration is impossible.

In [6]:
from fractions import Fraction

def h(n, j, lam=Fraction(0)):
    # h(n, j, lambda) as derived above, computed by the halving recursion
    if n == 1:
        assert j == 0
        return 1 + lam
    half = n // 2
    if j < half:
        return h(half, j, lam)
    return 2 * h(half, j - half, lam + Fraction(n, 4))

# agrees with brute force ...
for N in (2, 4, 8, 16):
    best = min_cost_by_valuation_bruteforce(N)
    assert all(h(N, j) == best[j][0] for j in range(N))
print("halving recursion matches brute force for N = 2, 4, 8, 16")

# ... and with the closed form, including at N = 64 where brute force is hopeless
assert all(h(64, j) == c_formula(j) for j in range(64))
print("halving recursion matches the closed form 2^p + j*2^(p-1) for all j < 64")
print("\nsample of c(j) for N = 64:")
for j in (0, 1, 2, 3, 31, 32, 47, 62, 63):
    print(f"  c({j:>2}) = {c_formula(j):>5}   (popcount {bin(j).count('1')})")

halving recursion matches brute force for N = 2, 4, 8, 16
halving recursion matches the closed form 2^p + j*2^(p-1) for all j < 64

sample of c(j) for N = 64:
  c( 0) =     1   (popcount 0)
  c( 1) =     3   (popcount 1)
  c( 2) =     4   (popcount 1)
  c( 3) =    10   (popcount 2)
  c(31) =   528   (popcount 5)
  c(32) =    34   (popcount 1)
  c(47) =   784   (popcount 5)
  c(62) =  1024   (popcount 5)
  c(63) =  2080   (popcount 6)


## 6. $N = 8$: the explicit script, and playing it against a cheating wheel

The schedule says round $k$ needs valuation $(N-1)-\nu_2(k)$; the previous section says the cheapest such move
is $u^j$. Concatenating gives a concrete 255-round script. We then verify it the honest way: propagate the set of
deviations the adversary can keep alive and confirm it is empty after round 255 — and *not* empty after 254,
so the script is tight.

In [7]:
def build_solution(N):
    best = min_cost_by_valuation_bruteforce(N) if N <= 16 else None
    reps = {j: (best[j][1] if best else u_power_mask(j, N)) for j in range(N)}
    return [reps[j] for j in valuation_schedule(N)]


def simulate(N, rounds_masks, report=()):
    # set of deviations the adversary can still keep alive; 0 means we have won
    alive = set(range(1, 1 << N))
    for t, S in enumerate(rounds_masks, start=1):
        alive = {d ^ rotate(S, r, N)
                 for d in alive for r in range(N)
                 if d ^ rotate(S, r, N) != 0}
        if t in report:
            print(f"  after round {t:>3}: {len(alive):>3} deviations still alive")
        if not alive:
            return True, t
    return False, len(rounds_masks)


N = 8
rounds8 = build_solution(N)
print(f"rounds       : {len(rounds8)}  (2^8 - 1 = {2**8 - 1})")
print(f"solution sum : {sum(cost(m, N) for m in rounds8)}")

print("\nadversarial simulation:")
won, when = simulate(N, rounds8, report=(1, 2, 4, 16, 64, 128, 254, 255))
print(f"  guaranteed win: {won} at round {when}")

won254, _ = simulate(N, rounds8[:-1])
print(f"  win by round 254 instead: {won254}  (the script is tight)")

answer_string = format_solution(rounds8, N)
print(f"\nsubmission string ({len(answer_string)} chars), first two rounds and last round:")
print("  " + ",".join(answer_string.split(",")[:16]) + ", ...")
print("  ... " + ",".join(answer_string.split(",")[-10:]))

rounds       : 255  (2^8 - 1 = 255)
solution sum : 6279

adversarial simulation:
  after round   1: 254 deviations still alive
  after round   2: 253 deviations still alive
  after round   4: 251 deviations still alive
  after round  16: 239 deviations still alive
  after round  64: 191 deviations still alive
  after round 128: 127 deviations still alive
  after round 254:   1 deviations still alive
  after round 255:   0 deviations still alive
  guaranteed win: True at round 255
  win by round 254 instead: False  (the script is tight)

submission string (3479 chars), first two rounds and last round:
  1,2,3,4,5,6,7,8,0,1,3,5,7,0,1,2, ...
  ... 0,1,2,3,4,5,6,7,8,0


## 7. The answers

Since valuation $j$ occurs $2^{\,j}$ times,

$$\text{optimal sum}(N) \;=\; \sum_{j=0}^{N-1} 2^{\,j}\Big(2^{p(j)} + j\,2^{\,p(j)-1}\Big), \qquad p(j)=\operatorname{popcount}(j).$$

Both $2^{\,j}$ and $2^{p(j)}$ factor over the binary digits of $j$, so for $N = 2^m$ the first half telescopes into a product:

$$\sum_{j<N} 2^{\,j}2^{p(j)} \;=\; \prod_{i=0}^{m-1}\Big(1 + 2^{\,2^i+1}\Big),$$

and differentiating that product in the usual generating-function way handles the $j\,2^{p-1}$ term. Handy as a
cross-check on a number with 23 digits.

In [8]:
def optimal_solution_sum(N):
    return sum((1 << j) * c_formula(j) for j in range(N))


def optimal_solution_sum_closed_form(m):
    # N = 2^m, via the product formula
    P = 1
    for i in range(m):
        P *= 1 + (1 << ((1 << i) + 1))
    weighted = 0
    for i in range(m):
        w = 1 << ((1 << i) + 1)
        weighted += (1 << i) * w * (P // (1 + w))
    return P + weighted // 2


print(f"{'N':>4} {'rounds':>22} {'optimal solution sum':>26}")
print("-" * 56)
for m in range(1, 7):
    N = 1 << m
    total = optimal_solution_sum(N)
    assert total == optimal_solution_sum_closed_form(m)
    print(f"{N:>4} {2**N - 1:>22} {total:>26}")

print()
assert optimal_solution_sum(2) == 7          # the example in the problem statement
assert optimal_solution_sum(4) == search_results[4][1]   # exhaustive search
assert optimal_solution_sum(8) == sum(cost(m, 8) for m in rounds8)
print("cross-checks passed: N=2 example, N=4 exhaustive search, N=8 explicit script")

a8, a64 = optimal_solution_sum(8), optimal_solution_sum(64)
print(f"\nANSWER       N = 8 : {a8}")
print(f"ANSWER (*)  N = 64 : {a64}")
print(f"                    ~ {a64 / 10**22:.6f} x 10^22   ({len(str(a64))} digits)")

   N                 rounds       optimal solution sum
--------------------------------------------------------
   2                      3                          7
   4                     15                        103
   8                    255                       6279
  16                  65535                    6262407
  32             4294967295              1619642912391
  64   18446744073709551615    27636190239652591799943

cross-checks passed: N=2 example, N=4 exhaustive search, N=8 explicit script

ANSWER       N = 8 : 6279
ANSWER (*)  N = 64 : 27636190239652591799943
                    ~ 2.763619 x 10^22   (23 digits)


## 8. Summary

1. $N$ is a power of two, so $\mathbb{F}_2[x]/(x^N-1) = \mathbb{F}_2[u]/(u^N)$ is **local**: its ideals form one chain
   $I_0 \supset I_1 \supset \cdots$.
2. Since $x^r S \equiv S \pmod{I_{v(S)+1}}$, **the spin cannot affect the leading level of a move**. The wheel
   degenerates into $N$ nested bits where flipping bit $j$ scrambles all higher bits.
3. Clearing such a state costs $T(N) = 2T(N-1)+1 = 2^N-1$ rounds, and the valuations are forced into the
   ruler sequence $v_k = (N-1)-\nu_2(k)$; valuation $j$ appears $2^{\,j}$ times.
4. Within a round, *any* subset of the required valuation works, so the label-sum minimises round by round.
   The cheapest subset of valuation $j$ is $u^j=(1+x)^j$, whose support is the submasks of $j$, of cost
   $2^{p} + j\,2^{p-1}$ — proved by the halving recursion and confirmed by brute force up to $N=16$.
5. Therefore $\displaystyle \text{optimal sum} = \sum_{j<N} 2^{\,j}\big(2^{p(j)} + j\,2^{\,p(j)-1}\big)$.

| $N$ | rounds | optimal solution sum |
|---:|---:|---:|
| 2 | 3 | 7 *(matches the statement)* |
| 4 | 15 | 103 *(matches exhaustive search)* |
| **8** | 255 | **6279** |
| 16 | 65535 | 373 ​419 |
| 32 | $2^{32}-1$ | 1 ​676 ​083 ​380 ​491 |
| **64** | $2^{64}-1$ | **27 ​636 ​190 ​239 ​652 ​591 ​799 ​943** |

### $N = 8$: `6279`
### $N = 64$ ($\star$): `27636190239652591799943`

The full 255-round script for $N=8$ is written to `answer_N8.txt` in IBM's comma-and-zero format by
`solve_wheel.py`; it begins `1,2,3,4,5,6,7,8,0,1,3,5,7,0,1,2,3,4,5,6,7,8,0,1,2,5,6,0,...`